### Q2

In [1]:
import io
import requests
import docx

In [2]:
def clean_line(line):
    line = line.strip()
    line = line.strip('\uFEFF')
    return line

def read_faq(file_id):
    url = f'https://docs.google.com/document/d/{file_id}/export?format=docx'
    
    response = requests.get(url)
    response.raise_for_status()
    
    with io.BytesIO(response.content) as f_in:
        doc = docx.Document(f_in)

    questions = []

    question_heading_style = 'heading 2'
    section_heading_style = 'heading 1'
    
    heading_id = ''
    section_title = ''
    question_title = ''
    answer_text_so_far = ''
     
    for p in doc.paragraphs:
        style = p.style.name.lower()
        p_text = clean_line(p.text)
    
        if len(p_text) == 0:
            continue
    
        if style == section_heading_style:
            section_title = p_text
            continue
    
        if style == question_heading_style:
            answer_text_so_far = answer_text_so_far.strip()
            if answer_text_so_far != '' and section_title != '' and question_title != '':
                questions.append({
                    'text': answer_text_so_far,
                    'section': section_title,
                    'question': question_title,
                })
                answer_text_so_far = ''
    
            question_title = p_text
            continue
        
        answer_text_so_far += '\n' + p_text
    
    answer_text_so_far = answer_text_so_far.strip()
    if answer_text_so_far != '' and section_title != '' and question_title != '':
        questions.append({
            'text': answer_text_so_far,
            'section': section_title,
            'question': question_title,
        })

    return questions

In [3]:
file_id = '1qZjwHkvP0lXHiE4zdbWyUXSVfmVGzougDD6N37bat3E'

In [4]:
faq_documents = {
    'llm-zoomcamp': file_id
}

documents = []

for course, file_id in faq_documents.items():
    print(course)
    course_documents = read_faq(file_id)
    documents.append({'course': course, 'documents': course_documents})

print(f"Number of FAQ documents processed: {len(documents)}")

llm-zoomcamp
Number of FAQ documents processed: 1


### Q3

In [5]:
import hashlib

def generate_document_id(doc):
    combined = f"{doc['course']}-{doc['question']}-{doc['text'][:10]}"
    hash_object = hashlib.md5(combined.encode())
    hash_hex = hash_object.hexdigest()
    document_id = hash_hex[:8]
    return document_id

In [6]:
def chunking_transform(documents):
    result_documents = []

    for course_dict in documents:
        for doc in course_dict['documents']:
            doc['course'] = course_dict['course']
            doc['document_id'] = generate_document_id(doc)
            result_documents.append(doc)

    print(f"Number of documents (chunks): {len(result_documents)}")
    return result_documents

In [7]:
result = chunking_transform(documents)

Number of documents (chunks): 86


### Q4

In [8]:
from datetime import datetime
from elasticsearch import Elasticsearch
import hashlib
import io
import requests
import docx

In [9]:
def clean_line(line):
    return line.strip().strip('\uFEFF')

def read_faq(file_id):
    url = f'https://docs.google.com/document/d/{file_id}/export?format=docx'
    response = requests.get(url)
    response.raise_for_status()
    
    with io.BytesIO(response.content) as f_in:
        doc = docx.Document(f_in)

    questions = []
    question_heading_style = 'heading 2'
    section_heading_style = 'heading 1'
    
    section_title = ''
    question_title = ''
    answer_text_so_far = ''
     
    for p in doc.paragraphs:
        style = p.style.name.lower()
        p_text = clean_line(p.text)
    
        if len(p_text) == 0:
            continue
    
        if style == section_heading_style:
            section_title = p_text
            continue
    
        if style == question_heading_style:
            answer_text_so_far = answer_text_so_far.strip()
            if answer_text_so_far != '' and section_title != '' and question_title != '':
                questions.append({
                    'text': answer_text_so_far,
                    'section': section_title,
                    'question': question_title,
                })
                answer_text_so_far = ''
    
            question_title = p_text
            continue
        
        answer_text_so_far += '\n' + p_text
    
    answer_text_so_far = answer_text_so_far.strip()
    if answer_text_so_far != '' and section_title != '' and question_title != '':
        questions.append({
            'text': answer_text_so_far,
            'section': section_title,
            'question': question_title,
        })

    return questions

def generate_document_id(doc):
    combined = f"{doc['course']}-{doc['question']}-{doc['text'][:10]}"
    hash_object = hashlib.md5(combined.encode())
    hash_hex = hash_object.hexdigest()
    return hash_hex[:8]

In [14]:
es = Elasticsearch(['http://localhost:9200'], verify_certs=False)

In [17]:
index_name_prefix = 'documents'
current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
index_name = f"{index_name_prefix}_{current_time}"
print("index name:", index_name)

index name: documents_20240816_162450


In [18]:
index_settings = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "section": {"type": "text"},
            "question": {"type": "text"},
            "course": {"type": "keyword"},
            "document_id": {"type": "keyword"}
        }
    }
}

In [19]:
es.indices.create(index=index_name, body=index_settings)

faq_documents = {
    'llm-zoomcamp': '1qZjwHkvP0lXHiE4zdbWyUXSVfmVGzougDD6N37bat3E'
}

documents = []
for course, file_id in faq_documents.items():
    course_documents = read_faq(file_id)
    documents.append({'course': course, 'documents': course_documents})

last_document = None

In [20]:
for course_dict in documents:
    for doc in course_dict['documents']:
        doc['course'] = course_dict['course']
        doc['document_id'] = generate_document_id(doc)
        es.index(index=index_name, id=doc['document_id'], body=doc)
        last_document = doc

print("Last document:")
print(last_document)

Last document:
{'text': 'Answer', 'section': 'Workshops: X', 'question': 'Question', 'course': 'llm-zoomcamp', 'document_id': 'd8c4c7bb'}


In [21]:
if last_document:
    print("Last document ID:", last_document['document_id'])
else:
    print("No documents were indexed")

Last document ID: d8c4c7bb


### Q5

In [22]:
query = {
    "query": {
        "match": {
            "text": "When is the next cohort?"
        }
    }
}

In [23]:
index_name = 'documents_20240816_162450'
response = es.search(index=index_name, body=query)

In [24]:
if response['hits']['hits']:
    top_hit = response['hits']['hits'][0]
    print(f"ID of the top matching result: {top_hit['_id']}")
    print(f"Score: {top_hit['_score']}")
    print(f"Content: {top_hit['_source']}")
else:
    print("No matching documents found.")

ID of the top matching result: a705279d
Score: 5.7542934
Content: {'text': "No, you can only get a certificate if you finish the course with a “live” cohort.\nWe don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your own project.\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.", 'section': 'General course-related questions', 'question': 'Certificate - Can I follow the course in a self-paced mode and get a certificate?', 'course': 'llm-zoomcamp', 'document_id': 'a705279d'}


### Q6

In [25]:

def clean_line(line):
    return line.strip().strip('\uFEFF')

def read_faq(file_id):
    url = f'https://docs.google.com/document/d/{file_id}/export?format=docx'
    response = requests.get(url)
    response.raise_for_status()
    
    with io.BytesIO(response.content) as f_in:
        doc = docx.Document(f_in)

    questions = []
    question_heading_style = 'heading 2'
    section_heading_style = 'heading 1'
    
    section_title = ''
    question_title = ''
    answer_text_so_far = ''
     
    for p in doc.paragraphs:
        style = p.style.name.lower()
        p_text = clean_line(p.text)
    
        if len(p_text) == 0:
            continue
    
        if style == section_heading_style:
            section_title = p_text
            continue
    
        if style == question_heading_style:
            answer_text_so_far = answer_text_so_far.strip()
            if answer_text_so_far != '' and section_title != '' and question_title != '':
                questions.append({
                    'text': answer_text_so_far,
                    'section': section_title,
                    'question': question_title,
                })
                answer_text_so_far = ''
    
            question_title = p_text
            continue
        
        answer_text_so_far += '\n' + p_text
    
    answer_text_so_far = answer_text_so_far.strip()
    if answer_text_so_far != '' and section_title != '' and question_title != '':
        questions.append({
            'text': answer_text_so_far,
            'section': section_title,
            'question': question_title,
        })

    return questions

def generate_document_id(doc):
    combined = f"{doc['course']}-{doc['question']}-{doc['text'][:10]}"
    hash_object = hashlib.md5(combined.encode())
    hash_hex = hash_object.hexdigest()
    return hash_hex[:8]

In [26]:
index_name_prefix = 'documents'
current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
index_name = f"{index_name_prefix}_{current_time}"
print("New index name:", index_name)

New index name: documents_20240816_163407


In [27]:
index_settings = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "section": {"type": "text"},
            "question": {"type": "text"},
            "course": {"type": "keyword"},
            "document_id": {"type": "keyword"}
        }
    }
}

In [28]:
es.indices.create(index=index_name, body=index_settings)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'documents_20240816_163407'})

In [29]:
faq_documents = {
    'llm-zoomcamp': '1T3MdwUvqCL3jrh3d3VCXQ8xE0UqRzI3bfgpfBq3ZWG0'
}

documents = []
for course, file_id in faq_documents.items():
    course_documents = read_faq(file_id)
    documents.append({'course': course, 'documents': course_documents})

In [30]:
last_document = None

for course_dict in documents:
    for doc in course_dict['documents']:
        doc['course'] = course_dict['course']
        doc['document_id'] = generate_document_id(doc)
        es.index(index=index_name, id=doc['document_id'], body=doc)
        last_document = doc

print("Indexing complete.")

Indexing complete.


In [31]:
query = {
    "query": {
        "match": {
            "text": "When is the next cohort?"
        }
    }
}

In [32]:
if response['hits']['hits']:
    top_hit = response['hits']['hits'][0]
    print(f"ID of the top matching result: {top_hit['_id']}")
    print(f"Score: {top_hit['_score']}")
    print(f"Content: {top_hit['_source']}")
else:
    print("No matching documents found.")

ID of the top matching result: a705279d
Score: 5.7542934
Content: {'text': "No, you can only get a certificate if you finish the course with a “live” cohort.\nWe don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your own project.\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.", 'section': 'General course-related questions', 'question': 'Certificate - Can I follow the course in a self-paced mode and get a certificate?', 'course': 'llm-zoomcamp', 'document_id': 'a705279d'}
